# **Model 3 - Pit Stop Prediction**

Models 1 and 2 were both regression, predicting continuous numbers. This one's different - we're 
predicting whether a lap is a pit lap or not, so it's a classification problem.

Target column is `is_pit_lap`, derived from `PitInTime`. Pit laps are rare compared to normal laps 
(around 3% of the dataset), so class imbalance is going to be a big part of getting this right.

Reusing the same cleanup steps from the last two notebooks (race sessions only, dropping red flag laps, 
deriving is_out_lap). Some features carry over from Models 1 and 2, but not all of them made sense for 
this problem, so only pulling in what's actually relevant to pit timing.

___
## **Initial | Imports | Checks | Target Derivation**

In [50]:
import pandas as pd
import numpy as np
from tqdm import tqdm

In [51]:
df = pd.read_parquet(r"C:\F1-AI\data\processed\fastf1_ml_base.parquet")
print(df.shape)

(469012, 78)


In [52]:
#Race only scope....
df = df[df['SessionName'] == 'Race'].copy()
print(df.shape)

(188482, 78)


In [53]:
# Red flag exclusion - as per anomalies detected while designing model 1.
df = df[df['status_red_flag'] == False].copy()
print(df.shape)

(188056, 78)


In [54]:
df['is_out_lap'] = df['PitOutTime'].notnull()
df['is_out_lap'].value_counts()

is_out_lap
False    182083
True       5973
Name: count, dtype: int64

**Target Derivation**

In [55]:
df['is_pit_lap'] = df['PitInTime'].notnull()
df['is_pit_lap'].value_counts(normalize=True)

is_pit_lap
False    0.969472
True     0.030528
Name: proportion, dtype: float64

___
## **Leakage Features Exclusion | Lagged features**

In [56]:
leakage_cols = [
    'LapTime', 'LapTime_seconds', 'Sector1Time', 'Sector2Time', 'Sector3Time',
    'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
    'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
    'Position', 'Deleted', 'DeletedReason',
    'FinalPosition', 'ClassifiedPosition', 'ResultTime', 'Status', 'Points', 'Laps',
    'PitInTime', 'PitOutTime'
]

df_model = df.drop(columns=leakage_cols)
print(df_model.shape)

(188056, 56)


In [57]:
df = df.sort_values(['Season', 'EventName', 'Driver', 'LapNumber'])

grp = df.groupby(['Season', 'EventName', 'Driver'])

df_model['driver_prev_lap_time'] = grp['LapTime_seconds'].shift(1)
df_model['position_prev_lap'] = grp['Position'].shift(1)

print(df_model[['driver_prev_lap_time', 'position_prev_lap']].isnull().sum())

driver_prev_lap_time    5920
position_prev_lap       3411
dtype: int64


In [58]:
n_groups = df_model.groupby(['Season', 'EventName', 'Driver']).ngroups
print(n_groups)

3411


## **Features**

In [59]:
grp_stint = df_model.groupby(['Season', 'EventName', 'Driver', 'Stint'])

df_model['stint_lap_number'] = grp_stint.cumcount() + 1

df_model['tyrelife_sqrt'] = np.sqrt(df_model['TyreLife'])
df_model['tyrelife_log1p'] = np.log1p(df_model['TyreLife'])

df_model['compound_tyrelife'] = df_model['Compound'].astype(str) + '_' + df_model['TyreLife'].astype(str)
df_model['compound_stint_lap'] = df_model['Compound'].astype(str) + '_' + df_model['stint_lap_number'].astype(str)

print(df_model[['stint_lap_number', 'tyrelife_sqrt', 'tyrelife_log1p']].describe())

       stint_lap_number  tyrelife_sqrt  tyrelife_log1p
count      187410.00000  186705.000000   186705.000000
mean           14.46545       3.639679        2.535806
std            10.66622       1.385271        0.757604
min             1.00000       1.000000        0.693147
25%             6.00000       2.645751        2.079442
50%            12.00000       3.605551        2.639057
75%            21.00000       4.582576        3.091042
max            77.00000       8.831761        4.369448


In [60]:
print(df_model['Stint'].isnull().sum())
print(df_model['TyreLife'].isnull().sum())

646
1351


In [61]:
null_stint_rows = df_model[df_model['Stint'].isnull()]
print(null_stint_rows[['Season', 'EventName', 'Driver', 'LapNumber', 'is_out_lap', 'is_pit_lap']].head(15))
print(null_stint_rows['is_out_lap'].value_counts())
print(null_stint_rows['is_pit_lap'].value_counts())

      Season              EventName Driver  LapNumber  is_out_lap  is_pit_lap
4299    2018  Australian_Grand_Prix    GAS        1.0       False       False
4300    2018  Australian_Grand_Prix    RAI        1.0       False       False
4301    2018  Australian_Grand_Prix    SAI        1.0       False       False
4302    2018  Australian_Grand_Prix    VET        1.0       False       False
4303    2018  Australian_Grand_Prix    HAM        1.0       False       False
4304    2018  Australian_Grand_Prix    SIR        1.0       False       False
4305    2018  Australian_Grand_Prix    VER        1.0       False       False
4306    2018  Australian_Grand_Prix    BOT        1.0       False       False
4307    2018  Australian_Grand_Prix    OCO        1.0       False       False
4308    2018  Australian_Grand_Prix    RIC        1.0       False       False
4310    2018  Australian_Grand_Prix    HUL        1.0       False       False
4311    2018  Australian_Grand_Prix    MAG        1.0       Fals

In [62]:
print(null_stint_rows['LapNumber'].value_counts())
print(null_stint_rows.groupby(['Season', 'EventName']).size())

LapNumber
1.0     291
2.0      31
3.0      15
4.0      15
5.0      15
6.0      15
7.0      15
8.0      15
9.0      15
10.0     15
11.0     15
12.0     15
13.0     15
14.0     15
15.0     15
16.0     15
17.0     15
18.0     15
19.0     15
20.0     15
21.0     15
22.0     15
23.0     15
24.0      9
Name: count, dtype: int64
Season  EventName            
2018    Australian_Grand_Prix     20
        Austrian_Grand_Prix       20
        Azerbaijan_Grand_Prix     14
        Bahrain_Grand_Prix        22
        Belgian_Grand_Prix         1
        British_Grand_Prix        21
        Canadian_Grand_Prix       17
        Chinese_Grand_Prix        21
        French_Grand_Prix         14
        German_Grand_Prix         21
        Hungarian_Grand_Prix      20
        Japanese_Grand_Prix       21
        Monaco_Grand_Prix         21
        Russian_Grand_Prix        21
        Singapore_Grand_Prix      20
        Spanish_Grand_Prix        18
2025    Miami_Grand_Prix         354
dtype: int64


In [63]:
non_lap1 = null_stint_rows[null_stint_rows['LapNumber'] != 1]
print(non_lap1.groupby(['Season', 'EventName'])['LapNumber'].agg(['min', 'max', 'count']))

miami = null_stint_rows[null_stint_rows['EventName'] == 'Miami_Grand_Prix']
print(miami['Driver'].value_counts())
print(miami['LapNumber'].min(), miami['LapNumber'].max())

                              min   max  count
Season EventName                              
2018   Australian_Grand_Prix  2.0   2.0      1
       Austrian_Grand_Prix    2.0   2.0      1
       Azerbaijan_Grand_Prix  2.0   2.0      1
       Bahrain_Grand_Prix     2.0   2.0      2
       British_Grand_Prix     2.0   2.0      1
       Canadian_Grand_Prix    2.0   2.0      1
       Chinese_Grand_Prix     2.0   2.0      1
       French_Grand_Prix      2.0   2.0      1
       German_Grand_Prix      2.0   2.0      1
       Hungarian_Grand_Prix   2.0   2.0      1
       Japanese_Grand_Prix    2.0   2.0      1
       Monaco_Grand_Prix      2.0   2.0      1
       Russian_Grand_Prix     2.0   2.0      1
       Singapore_Grand_Prix   2.0   2.0      1
       Spanish_Grand_Prix     2.0   2.0      1
2025   Miami_Grand_Prix       2.0  24.0    339
Driver
VER    24
NOR    24
PIA    24
ALB    24
ANT    24
LEC    24
TSU    24
SAI    24
RUS    24
GAS    23
HUL    23
ALO    23
HAM    23
BEA    23
LAW    

In [64]:
df_model = df_model.sort_values(['Season', 'EventName', 'Driver', 'LapNumber'])

df_model['Stint'] = df_model.groupby(['Season', 'EventName', 'Driver'])['Stint'].ffill()
df_model['Stint'] = df_model['Stint'].fillna(1)

print(df_model['Stint'].isnull().sum())

0


In [65]:
grp_stint = df_model.groupby(['Season', 'EventName', 'Driver', 'Stint'])

df_model['stint_lap_number'] = grp_stint.cumcount() + 1

df_model['tyrelife_sqrt'] = np.sqrt(df_model['TyreLife'])
df_model['tyrelife_log1p'] = np.log1p(df_model['TyreLife'])

df_model['compound_tyrelife'] = df_model['Compound'].astype(str) + '_' + df_model['TyreLife'].astype(str)
df_model['compound_stint_lap'] = df_model['Compound'].astype(str) + '_' + df_model['stint_lap_number'].astype(str)

print(df_model['stint_lap_number'].isnull().sum())
print(df_model[['tyrelife_sqrt', 'tyrelife_log1p']].describe())

0
       tyrelife_sqrt  tyrelife_log1p
count  186705.000000   186705.000000
mean        3.639679        2.535806
std         1.385271        0.757604
min         1.000000        0.693147
25%         2.645751        2.079442
50%         3.605551        2.639057
75%         4.582576        3.091042
max         8.831761        4.369448


#### **SC/VSC Recovery features from model 2**

In [66]:
df_model['is_clean_green_lap'] = (
    df_model['status_green'] &
    ~df_model['status_safety_car'] &
    ~df_model['status_vsc'] &
    ~df_model['status_yellow'] &
    ~df_model['status_red_flag']
)

df_model = df_model.sort_values(['Season', 'EventName', 'LapNumber'])

df_model['is_restart_lap'] = (
    df_model.groupby(['Season', 'EventName'])['is_clean_green_lap']
    .transform(lambda x: x & ~x.shift(1, fill_value=False))
)

print(df_model['is_clean_green_lap'].value_counts())
print(df_model['is_restart_lap'].sum())

is_clean_green_lap
True     166035
False     22021
Name: count, dtype: int64
2284


#### **SC REcovery decay**

In [67]:
df_model['restart_lap_number'] = df_model['LapNumber'].where(df_model['is_restart_lap'])

df_model['restart_lap_number'] = (
    df_model.groupby(['Season', 'EventName'])['restart_lap_number'].ffill()
)

df_model['laps_since_green_resumed'] = df_model['LapNumber'] - df_model['restart_lap_number']
df_model['sc_recovery_decay'] = np.exp(-df_model['laps_since_green_resumed'].fillna(99) / 3)

print(df_model['laps_since_green_resumed'].describe())

count    182170.000000
mean         16.185140
std          15.482417
min           0.000000
25%           4.000000
50%          11.000000
75%          25.000000
max          77.000000
Name: laps_since_green_resumed, dtype: float64


#### **Fuel Load proxy**

In [68]:
max_lap = df_model.groupby(['Season', 'EventName'])['LapNumber'].transform('max')
df_model['fuel_load_pct'] = 1 - (df_model['LapNumber'] - 1) / max_lap
df_model['fuel_load_proxy'] = df_model['fuel_load_pct'] * 110

#### **Race progress**

In [69]:
df_model['race_progress_pct'] = (df_model['LapNumber'] - 1) / max_lap

#### **Commulative green laps**

In [70]:
df_model['cumulative_green_laps'] = (
    df_model.groupby(['Season', 'EventName', 'Driver'])['is_clean_green_lap'].cumsum()
)
df_model['green_laps_sqrt'] = np.sqrt(df_model['cumulative_green_laps'])

#### **Position based features**

In [71]:
df_model['is_race_leader'] = df_model['position_prev_lap'] == 1

df_model['grid_delta'] = df_model['position_prev_lap'] - df_model['GridPosition']

#### **Lagged Weather**

In [72]:
grp = df_model.groupby(['Season', 'EventName', 'Driver'])

for col in ['Rainfall', 'AirTemp', 'TrackTemp', 'Humidity']:
    df_model[f'prev_lap_{col.lower()}'] = grp[col].shift(1)

df_model['track_temp_minus_airtemp'] = df_model['prev_lap_tracktemp'] - df_model['prev_lap_airtemp']

#### **Laps to go and mandatory two compound flag (AI Sugegstion)**

In [73]:
df_model['laps_to_go'] = max_lap - df_model['LapNumber']

compound_count_per_race = (
    df_model.groupby(['Season', 'EventName', 'Driver'])['Compound'].transform('nunique')
)
race_level_multi_compound = (
    df_model.assign(used_multi=compound_count_per_race >= 2)
    .groupby(['Season', 'EventName'])['used_multi'].transform('mean')
)
df_model['is_mandatory_two_compound_race'] = race_level_multi_compound > 0.95

#### **TYpical stint length by compound and track, tyre life pct**

In [74]:
stint_lengths = (
    df_model[df_model['is_pit_lap']]
    .groupby(['EventName', 'Compound'])['TyreLife']
    .quantile(0.75)
    .rename('typical_stint_length')
)

df_model = df_model.merge(stint_lengths, on=['EventName', 'Compound'], how='left')
df_model['tyre_life_pct_of_typical'] = df_model['TyreLife'] / df_model['typical_stint_length']

print(df_model['typical_stint_length'].isnull().sum())

1188


In [75]:
compound_fallback = (
    df_model[df_model['is_pit_lap']]
    .groupby('Compound')['TyreLife']
    .quantile(0.75)
)

df_model['typical_stint_length'] = df_model['typical_stint_length'].fillna(
    df_model['Compound'].map(compound_fallback)
)

df_model['tyre_life_pct_of_typical'] = df_model['TyreLife'] / df_model['typical_stint_length']

print(df_model['typical_stint_length'].isnull().sum())

686


In [76]:
still_null = df_model[df_model['typical_stint_length'].isnull()]
print(still_null['Compound'].value_counts(dropna=False))
print(still_null[['Season', 'EventName']].drop_duplicates())

Compound
nan        646
UNKNOWN     40
Name: count, dtype: int64
        Season              EventName
943       2018  Australian_Grand_Prix
1883      2018    Austrian_Grand_Prix
3128      2018  Azerbaijan_Grand_Prix
3975      2018     Bahrain_Grand_Prix
4991      2018     Belgian_Grand_Prix
6984      2018     British_Grand_Prix
7886      2018    Canadian_Grand_Prix
9105      2018     Chinese_Grand_Prix
10222     2018      French_Grand_Prix
11140     2018      German_Grand_Prix
12392     2018   Hungarian_Grand_Prix
13625     2018    Japanese_Grand_Prix
15847     2018      Monaco_Grand_Prix
17363     2018     Russian_Grand_Prix
18311     2018   Singapore_Grand_Prix
19457     2018     Spanish_Grand_Prix
67660     2021     Belgian_Grand_Prix
178911    2025       Miami_Grand_Prix


umhhh ... XGBoost will handle it , i am leaving this...

#### **Rain increasing and First lap after SC**

In [77]:
df_model['is_rain_increasing'] = df_model['prev_lap_rainfall'] > df_model.groupby(
    ['Season', 'EventName', 'Driver']
)['prev_lap_rainfall'].shift(1)

df_model['is_first_lap_after_sc'] = df_model['is_restart_lap']

#### **Position change and cross-driver leader-pit feature (Very impotant feature i think)**

In [78]:
grp = df_model.groupby(['Season', 'EventName', 'Driver'])
df_model['position_two_laps_ago'] = grp['position_prev_lap'].shift(1)
df_model['position_change_last_lap'] = df_model['position_two_laps_ago'] - df_model['position_prev_lap']

In [79]:
leader_pit_per_lap = (
    df_model[df_model['position_prev_lap'] == 1]
    .groupby(['Season', 'EventName', 'LapNumber'])['is_pit_lap']
    .max()
    .rename('leader_pitted_this_lap')
)

df_model = df_model.merge(
    leader_pit_per_lap, on=['Season', 'EventName', 'LapNumber'], how='left'
)

df_model['leader_pitted_this_lap'] = np.where(
    df_model['leader_pitted_this_lap'].isna(), False, df_model['leader_pitted_this_lap']
).astype(bool)

df_model['leader_pitted_prev_lap'] = (
    df_model.sort_values(['Season', 'EventName', 'LapNumber'])
    .groupby(['Season', 'EventName'])['leader_pitted_this_lap']
    .shift(1)
)

print(df_model['leader_pitted_prev_lap'].value_counts(dropna=False))

leader_pitted_prev_lap
False    179844
True       8040
NaN         172
Name: count, dtype: int64


In [80]:
print(df_model.groupby(['Season', 'EventName']).ngroups)

172


___
## **FEATURE SELECTION AND DROPS**

## Getting the feature list ready

Feature engineering is basically done. Before jumping into modeling, need to sort columns into 
three buckets - stuff that's purely an identifier and should be dropped, categorical stuff that 
needs encoding, and numeric/boolean stuff that's already model-ready.

In [84]:
pd.set_option('display.max_rows', None)
print(df_model.shape)
df_model.dtypes.sort_values()

(188056, 90)


is_pit_lap                                   bool
is_race_leader                               bool
status_vsc                                   bool
status_vsc_ending                            bool
status_unknown                               bool
is_out_lap                                   bool
status_unused3                               bool
status_yellow                                bool
status_green                                 bool
is_mandatory_two_compound_race               bool
IsAccurate                                   bool
status_red_flag                              bool
FastF1Generated                              bool
leader_pitted_this_lap                       bool
is_rain_increasing                           bool
FreshTyre                                    bool
is_first_lap_after_sc                        bool
is_clean_green_lap                           bool
is_restart_lap                               bool
status_safety_car                            bool


In [86]:
drop_cols = [
    'Driver', 'DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId',
    'FirstName', 'LastName', 'FullName', 'HeadshotUrl', 'CountryCode', 'TeamColor', 'TeamId',
    'SessionName',
    'Time', 'LapStartTime', 'LapStartDate', 'Q1', 'Q2', 'Q3',
    'restart_lap_number',
    'AirTemp', 'Humidity', 'TrackTemp', 'Rainfall',
    'TrackStatus',
    'GridPosition'
]

df_model = df_model.drop(columns=drop_cols)
print(df_model.shape)

(188056, 64)


In [87]:
df_model['leader_pitted_prev_lap'] = np.where(
    df_model['leader_pitted_prev_lap'].isna(), False, df_model['leader_pitted_prev_lap']
).astype(bool)

print(df_model['leader_pitted_prev_lap'].value_counts())

leader_pitted_prev_lap
False    180016
True       8040
Name: count, dtype: int64


In [88]:
for col in ['Compound', 'Compound_category', 'Team', 'EventName', 'compound_tyrelife', 'compound_stint_lap']:
    print(col, df_model[col].nunique())

Compound 11
Compound_category 6
Team 19
EventName 36
compound_tyrelife 518
compound_stint_lap 519


In [89]:
df_model = df_model.drop(columns=['compound_tyrelife', 'compound_stint_lap'])
print(df_model.shape)

(188056, 62)


In [90]:
df_model = pd.get_dummies(df_model, columns=['Compound', 'Compound_category', 'Team', 'EventName'], drop_first=False)
print(df_model.shape)

(188056, 130)


___
## **Train/Test Split**


Splitting by race instead of randomly. If laps from the same race end up in both train and test, 
the model could learn race-specific quirks (like exact SC timing) that won't generalize, and the 
validation score would look better than it actually is.

In [93]:
race_keys = df_model[['Season', 'EventName_' + df_model.filter(like='EventName_').columns[0].split('EventName_')[1]]]

In [94]:
print(df_model.columns[df_model.columns.str.contains('EventName') | df_model.columns.str.contains('Season')].tolist()[:5])

['Season', 'EventName_70th_Anniversary_Grand_Prix', 'EventName_Abu_Dhabi_Grand_Prix', 'EventName_Australian_Grand_Prix', 'EventName_Austrian_Grand_Prix']
